# Beautiful Soup


Parsing basic HTML content from webpages and extracting the information we need is what beautiful soup is designed for.

### Python Jobs

In [ ]:
%pip install bs4

In [ ]:
import requests
from bs4 import BeautifulSoup

# Go to http://pythonjobs.github.io/ and press ctrl + shift + I or f12 and start inspecting the elements in the page

URL = 'http://pythonjobs.github.io/'
page = requests.get(URL)

soup = BeautifulSoup(page.content, 'html.parser')

In [ ]:
# The raw html of the page
print(page.text)

# You can try the code below if the simple print above looks ugly in your environment
# import pprint
# pp = pprint.PrettyPrinter(indent=4)
# pp.pprint(page.text)

In [ ]:
# Get the list-title element from the page
elem = soup.find(id='list-title')
print(elem)

In [ ]:
# Get the sections that have the "job_list" class (should be 1 element - the list of jobs)
section = soup.find_all("section", {"class": "job_list"})
print(section)

### Exercise 1 - Warmup
Get a list with all the jobs from the main page.
For each job print the title, the short description and the hyperlink to the full job description.

Hints:
Jobs appear to be divs that have class = "job".

For extracted elements (such as a div) you can use the .find() method. E.g. my_elem.find('p') should give you a p (p is the paragraph tag in html) element that is beneath my_elem in the html tree structure.

To access the text of an element you can use element.text. 

To access the href of an element you can use element['href']


In [ ]:
jobs = section[0].find_all("div", {"class": "job"})

for job in jobs:
    title = job.find("h1").find("a").text
    description = job.find("p", {"class": "detail"}).text
    link = job.find("a")["href"]
    
    print(f"Title: {title}")
    print(f"Description: {description}")
    print(f"Link: {URL}{link}\n")

### Politifact

In [ ]:
import urllib.request, sys, time
import pandas as pd

pagesToGet = 1

upperframe = []  
for page in range(1, pagesToGet + 1):
    print('processing page :', page)
    url = f'https://www.politifact.com/factchecks/list/?page={page}'
    print(url)
    
    # an exception might be thrown, so the code should be in a try-except block
    try:
        # use the browser to get the url. This is a suspicious command that might blow up.
        page = requests.get(url)                              # this might throw an exception if something goes wrong.
    
    except Exception as e:                                    # this describes what to do if an exception is thrown
        error_type, error_obj, error_info = sys.exc_info()    # get the exception information
        print ('ERROR FOR LINK:', url)                         # print the link that cause the problem
        print (error_type, 'Line:', error_info.tb_lineno)     # print error info and line that threw the exception
        continue                                              # ignore this page. Abandon this and go back.
    time.sleep(2)   
    soup = BeautifulSoup(page.text, 'html.parser')
    frame = []
    links = soup.find_all('li', attrs={'class':'o-listicle__item'})
    print(len(links))
    filename = "NEWS.csv"
    f = open(filename,"w", encoding = 'utf-8')
    headers="Statement,Link, Source\n"
    f.write(headers)
    
    for link in links:
        Statement = link.find("div", attrs={'class':'m-statement__quote'}).text.strip()
        Link = "https://www.politifact.com"
        Link += link.find("div",attrs={'class':'m-statement__quote'}).find('a')['href'].strip()
        Source = link.find('div', attrs={'class':'m-statement__meta'}).find('a').text.strip()
        frame.append((Statement, Link, Source))
        
        f.write(f'{Statement.replace(",","^")},{Link},{Source.replace(",","^")}\n')
    upperframe.extend(frame)
    
f.close()

data = pd.DataFrame(upperframe, columns=['Statement', 'Link', 'Source'])

In [ ]:
data

### Exercise 2
Scrape the date and the label (false, true, etc.) information for each statement as well.

Create a new dataframe where you have the Statement, Link, **Date**, Source and **Label** columns for the first page of politifact statements.

Hint: The 'alt' attribute of an image can come in handy


In [ ]:
import re

pagesToGet = 1

frame = []
for page in range(1, pagesToGet + 1):
    url = f'https://www.politifact.com/factchecks/list/?page={page}'
    
    try:
        page = requests.get(url)
        
    except Exception as e:
        error_type, error_obj, error_info = sys.exc_info()
        print('ERROR FOR LINK:', url)
        print(error_type, 'Line:', error_info.tb_lineno)
        continue
    
    soup = BeautifulSoup(page.text, 'html.parser')
    posts = soup.find_all('li', attrs={'class':'o-listicle__item'})
    
    for post in posts:
        statement = post.find("div", attrs={'class':'m-statement__quote'}).text.strip()
        link = "https://www.politifact.com" + post.find("div", attrs={'class':'m-statement__quote'}).find('a')['href'].strip()
        source = post.find('div', attrs={'class':'m-statement__meta'}).find('a').text.strip()
        date = re.search(r"[a-zA-Z]+ [0-9]{1,2}, [0-9]{4}", post.find('div', {'class': 'm-statement__desc'}).text).group(0)
        label = post.find('article')['class'][2].split('--')[1]
        
        frame += [(statement, link, source, date, label)]

data = pd.DataFrame(frame, columns=['statement', 'link', 'source', 'date', 'label'])
data

### Weather

In [ ]:
page = requests.get("https://forecast.weather.gov/MapClick.php?lat=37.7772&lon=-122.4168")
soup = BeautifulSoup(page.content, 'html.parser')
seven_day = soup.find(id="seven-day-forecast")
forecast_items = seven_day.find_all(class_="tombstone-container")
tonight = forecast_items[0]
print(tonight.prettify())

In [ ]:
period = tonight.find(class_="period-name").get_text()
short_desc = tonight.find(class_="short-desc").get_text()
temp = tonight.find(class_="temp").get_text()
print(period)
print(short_desc)
print(temp)

In [ ]:
img = tonight.find("img")
desc = img['title']
print(desc)

In [ ]:
period_tags = seven_day.select(".tombstone-container .period-name")
periods = [pt.get_text() for pt in period_tags]
periods

In [ ]:
short_descs = [sd.get_text() for sd in seven_day.select(".tombstone-container .short-desc")]
temps = [t.get_text() for t in seven_day.select(".tombstone-container .temp")]
descs = [d["title"] for d in seven_day.select(".tombstone-container img")]
print(short_descs)
print(temps)
print(descs)

In [ ]:
import pandas as pd
weather = pd.DataFrame({
    "period": periods,
    "short_desc": short_descs,
    "temp": temps,
    "desc":descs
})
weather

### Exercise 3 - Basic pandas revision



1. Calculate the average temperature for the data that you have obtained


In [ ]:
def temp_to_num(temp: str) -> int:
    fahrenheit = temp.split()[1]
    return int(fahrenheit)

weather['temp'].map(temp_to_num).mean()


2. Calculate the average night temperature for the data that you have obtained


In [ ]:
weather.set_index('period').filter(like="Night", axis=0)['temp'].map(temp_to_num).mean()


3. Display all of the periods when it's expected to be sunny for the data that you have obtained


In [ ]:
weather[weather['short_desc'].str.contains("Sunny")]['period']

# Selenium

For non-static websites (sites that are interactive e.g. with the use of javascript) it is usually not sufficient to use beautiful soup for parsing. 

Selenium is a tool that can mimic and automate human tasks (such as clicking, hovering, etc.) as well as allowing element selection on webpages. *The main workhorse behind the library is the Webdriver* which needs to be installed.

If you are not using Google Colab, then try installing the chrome web driver either using anaconda: conda install -c conda-forge python-chromedriver-binary or downloading the appropriate webdriver for your chrome version. Ask in the laboratory for more details if needed.

We suggest using Google Colab for this laboratory to avoid the local web driver installation step.


In [ ]:
# below commands are for google colab, see above text for more info on jupyter notebook version
%pip install selenium
%pip install webdriver-manager


In [ ]:
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager

options = webdriver.ChromeOptions()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

driver = webdriver.Chrome(ChromeDriverManager().install(), options=options)


In [ ]:
from selenium.webdriver.common.by import By

def scrape_nba_salaries(year_from, year_to):
    df = pd.DataFrame(columns=['Player','Salary','Year']) # creates master dataframe 

    for year in range(year_from, year_to):
        url = f'https://hoopshype.com/salaries/players/{year}-{year + 1}/'
        print(url)
        driver.get(url)
        
        players = driver.find_elements(by=By.XPATH, value='//td[@class="name"]')
        salaries = driver.find_elements(by=By.XPATH, value='//td[@class="hh-salaries-sorted"]') 
        
        players_list = []
        for player in players:
            if player.text.strip():
                players_list += [player.text]
        
        salaries_list = []
        for salarie in salaries:
            salaries_list += [salarie.text]
        
        data_tuples = list(zip(players_list[1:], salaries_list[1:])) # list of each player's name and salary paired together
        
        temp_df = pd.DataFrame(data_tuples, columns=['Player','Salary'])
        temp_df['Year'] = year
        
        df = pd.concat([df, temp_df])
        
    return df

### Exercise 4



1. Calculate the average salary of an NBA player in 1990-1991 and the average salary of an NBA player in 2021-2022


In [ ]:
df_old = scrape_nba_salaries(1990, 1991)
df_new = scrape_nba_salaries(2020, 2021)

In [ ]:
print(df_old)
print(df_new)

In [ ]:
avg_salary_old = df_old['Salary'].map(lambda x: int(x.replace('$','').replace(',',''))).mean()
avg_salary_new = df_new['Salary'].map(lambda x: int(x.replace('$','').replace(',',''))).mean()

print(f'Average NBA salary from 1990-1991: ${avg_salary_old:,.0f}')
print(f'Average NBA salary from 2021-2022: ${avg_salary_new:,.0f}')


2. Create a dataframe with the team payrolls from 1999-2000, 2010-2011 and 2021-2022. Calculate the average salary of the top 10 teams in each year.


In [ ]:
def scrape_nba_team_payrolls(year_from, year_to):
    df = pd.DataFrame(columns=['Team','Payroll','Year']) # creates master dataframe 

    for year in range(year_from, year_to):
        url = f'https://hoopshype.com/salaries/{year}-{year + 1}/'
        print(url)
        driver.get(url)
        
        teams = driver.find_elements(by=By.XPATH, value='//td[@class="name"]')
        payrolls = driver.find_elements(by=By.XPATH, value='//td[@class="hh-salaries-sorted"]') 
        
        teams_list = []
        for team in teams:
            if team.text.strip():
                teams_list += [team.text]
        
        payrolls_list = []
        for payroll in payrolls:
            payrolls_list += [payroll.text]
        
        data_tuples = list(zip(teams_list[1:], payrolls_list[1:])) # list of each team's name and payroll paired together
        
        temp_df = pd.DataFrame(data_tuples, columns=['Team','Payroll'])
        temp_df['Year'] = year
        
        df = pd.concat([df, temp_df])
        
    return df

In [ ]:
team_payroll_old = scrape_nba_team_payrolls(1999, 2000)
team_payroll_mid = scrape_nba_team_payrolls(2010, 2011)
team_payroll_new = scrape_nba_team_payrolls(2020, 2021)

In [ ]:
print(team_payroll_old)

In [ ]:
def dollar_to_int(dollar: str):
    return int(dollar.replace('$','').replace(',',''))

avg_payroll_old = team_payroll_old.head(10)['Payroll'].map(dollar_to_int).mean()
avg_payroll_mid = team_payroll_mid.head(10)['Payroll'].map(dollar_to_int).mean()
avg_payroll_new = team_payroll_new.head(10)['Payroll'].map(dollar_to_int).mean()

print(f'Average NBA payroll from 1999-2000: ${avg_payroll_old:,.0f}')
print(f'Average NBA payroll from 2010-2011: ${avg_payroll_mid:,.0f}')
print(f'Average NBA payroll from 2021-2022: ${avg_payroll_new:,.0f}')


3. Create a dataframe containing the 10 best paid NBA players for each of the 5 years from 2017-2018 to 2021-2022. This means you will have 50 entries in your dataframe.

    Display all of the players that appear more than 3 times in the dataframe (meaning they were in top 10 for at least 3 out of the 5 years). 
    Hint: .value_counts() method may be of use


In [ ]:
def scrape_top_nba_player_salaries(year_from, year_to):
    df = pd.DataFrame(columns=['Player','Salary','Year']) # creates master dataframe 

    for year in range(year_from, year_to):
        url = f'https://hoopshype.com/salaries/players/{year}-{year + 1}/'
        print(url)
        driver.get(url)
        
        players = driver.find_elements(by=By.XPATH, value='//td[@class="name"]')
        salaries = driver.find_elements(by=By.XPATH, value='//td[@class="hh-salaries-sorted"]')[1:11] 
        
        players_list = [player.text for player in players if player.text.strip()][1:11]
        salaries_list = [salary.text for salary in salaries]
        
        data_tuples = list(zip(players_list, salaries_list))
        
        temp_df = pd.DataFrame(data_tuples, columns=['Player','Salary'])
        temp_df['Year'] = year
        
        df = pd.concat([df, temp_df])
        
    return df

In [ ]:
best_paid_players = scrape_top_nba_player_salaries(2016, 2021)
best_paid_players

In [ ]:
best_paid_players_count = count(best_paid_players, 'Player', sort=True)
best_paid_players_count[best_paid_players_count['n'] >= 3]

# News Articles

In [ ]:
%pip install newspaper3k

In [ ]:
# Articles
import nltk
nltk.download('punkt')
from newspaper import Article

article = Article('https://www.digi24.ro/stiri/actualitate/politica/klaus-iohannis-la-intalnirea-cu-premierul-belgiei-trebuie-sa-asiguram-un-sprijin-substantial-si-coerent-pentru-republica-moldova-1903405')
article.download()
article.parse()
article.nlp()

In [ ]:
article.text

In [ ]:
article.publish_date

In [ ]:
article.keywords

In [ ]:
article.summary

In [ ]:
# Let's get some financial news from marketwatch.com
import newspaper
from newspaper import Config
from newspaper import Article, ArticleException

USER_AGENT = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:78.0) Gecko/20100101 Firefox/78.0'

config = Config()
config.browser_user_agent = USER_AGENT
config.request_timeout = 10

base_url = 'https://www.marketwatch.com'
article_urls = set()
marketwatch = newspaper.build(base_url, config=config, memoize_articles=False, language='en')

for sub_article in marketwatch.articles:
    article = Article(sub_article.url, config=config, memoize_articles=False, language='en')
    article.download()
    
    try:
        article.parse()
    except:
        print("Article not found for ", sub_article.url)
        
    if article.url not in article_urls:
        article_urls.add(article.url)

    # The majority of the article elements are located
    # within the meta data section of the page's
    # navigational structure
    article_meta_data = article.meta_data

    published_date = {value for (key, value) in article_meta_data.items() if key == 'parsely-pub-date'}
    article_published_date = " ".join(str(x) for x in published_date)

    authors = sorted({value for (key, value) in article_meta_data.items() if key == 'parsely-author'})
    article_author = ', '.join(authors)

    title = {value for (key, value) in article_meta_data.items() if key == 'parsely-title'}
    article_title = " ".join(str(x) for x in title)

    keywords = ''.join({value for (key, value) in article_meta_data.items() if key == 'keywords'})
    keywords_list = sorted(keywords.lower().split(','))
    article_keywords = ', '.join(keywords_list)

    tags = ''.join({value for (key, value) in article_meta_data.items() if key == 'parsely-tags'})
    tag_list = sorted(tags.lower().split(','))
    article_tags = ', '.join(tag_list)

    summary = {value for (key, value) in article_meta_data.items() if key == 'description'}
    article_summary = " ".join(str(x) for x in summary)

    # the replace is used to remove newlines
    article_text = article.text.replace('\n', '')
    # print(article_text)
    print(article_summary)

### Exercise 5

Let's use our naive afinn lexicon method for sentiment analysis on the latest marketwatch.com news articles.


In [ ]:
class Color:
    GREEN = '\033[92m'
    RED = '\033[91m'
    MAGENTA = '\033[95m'
    END = '\033[0m'

In [ ]:
from afinn import Afinn
import newspaper
from newspaper import Config
from newspaper import Article, ArticleException

afinn = Afinn()

USER_AGENT = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:78.0) Gecko/20100101 Firefox/78.0'

config = Config()
config.browser_user_agent = USER_AGENT
config.request_timeout = 10

base_url = 'https://www.marketwatch.com'
article_urls = set()
marketwatch = newspaper.build(base_url, config=config, memoize_articles=False, language='en')


1. Display the first 30 titles and their afinn score. (apply afinn only on the title)


In [ ]:
article_count = 0
for sub_article in marketwatch.articles:
    if article_count > 10:
        break
    
    article = Article(sub_article.url, config=config, memoize_articles=False, language='en')
    article.download()
    
    try:
        article.parse()
    except:
        print("Article not found for ", sub_article.url)

    article_meta_data = article.meta_data

    title = {value for (key, value) in article_meta_data.items() if key == 'parsely-title'}
    article_title = " ".join(str(x) for x in title)
    
    print(f'{article_title} - score: {Color.MAGENTA}{afinn.score(article_title)}{Color.END}')
    
    article_count += 1


2. Display the first 30 titles and their corresponding article text afinn score. (apply afinn on the text of the article)


In [ ]:
article_count = 0
for sub_article in marketwatch.articles:
    if article_count > 10:
        break
    
    article = Article(sub_article.url, config=config, memoize_articles=False, language='en')
    article.download()
    
    try:
        article.parse()
    except:
        print("Article not found for ", sub_article.url)

    article_meta_data = article.meta_data

    title = {value for (key, value) in article_meta_data.items() if key == 'parsely-title'}
    article_title = " ".join(str(x) for x in title)
    
    print(f'{article_title} - score: {Color.MAGENTA}{afinn.score(article.text)}{Color.END}')
    
    article_count += 1


3. Display the title of the first 20 clearly positive articles (we'll say an article is clearly positive if its afinn score is >= 10)


In [ ]:
article_count = 0
for sub_article in marketwatch.articles:
    if article_count >= 20:
        break
    
    article = Article(sub_article.url, config=config, memoize_articles=False, language='en')
    article.download()
    
    try:
        article.parse()
    except:
        pass
        
    score = afinn.score(article.text)
    
    if score < 10:
        continue

    article_meta_data = article.meta_data

    title = {value for (key, value) in article_meta_data.items() if key == 'parsely-title'}
    article_title = " ".join(str(x) for x in title)
    
    print(f'{article_title} - score: {Color.GREEN}{afinn.score(article.text)}{Color.END}')
    
    article_count += 1


4. Display the title of the first 20 clearly negative articles (we'll say an article is clearly positive if its afinn score is <= -10)


In [ ]:
article_count = 0
for sub_article in marketwatch.articles:
    if article_count >= 20:
        break
    
    article = Article(sub_article.url, config=config, memoize_articles=False, language='en')
    article.download()
    
    try:
        article.parse()
    except:
        pass
        
    score = afinn.score(article.text)
    
    if score > -10:
        continue

    article_meta_data = article.meta_data

    title = {value for (key, value) in article_meta_data.items() if key == 'parsely-title'}
    article_title = " ".join(str(x) for x in title)
    
    print(f'{article_title} - score: {Color.RED}{afinn.score(article.text)}{Color.END}')
    
    article_count += 1

# Reddit

In [ ]:
%pip install praw

In [ ]:
import praw
reddit = praw.Reddit(client_id='sIXZTihLNiKiHw',
                     client_secret='EjfAsmz5z8mDbZohe4UPYTPIZsYmOQ',
                     user_agent='Test1', check_for_async=False)

In [ ]:
# Get the first 10 hot posts on the main page of reddit (/all)
hot_posts = reddit.subreddit('all').hot(limit=10)
for post in hot_posts:
    print(post.title, post.score)

In [ ]:
import pandas as pd

posts = []
ml_subreddit = reddit.subreddit('MachineLearning')
for post in ml_subreddit.hot(limit=10):
    posts.append([post.title, post.score, post.id, post.subreddit, post.url, post.num_comments, post.selftext, post.created])
posts = pd.DataFrame(posts,columns=['title', 'score', 'id', 'subreddit', 'url', 'num_comments', 'body', 'created'])
posts

In [ ]:
# Getting a particular post by url or by id
submission = reddit.submission(url="https://www.reddit.com/r/MapPorn/comments/a3p0uq/an_image_of_gps_tracking_of_multiple_wolves_in/")
# or 
submission = reddit.submission(id="a3p0uq") #id comes after comments/

In [ ]:
# Accessing the comments of a post
for top_level_comment in submission.comments:
    try:
        print(top_level_comment.body, top_level_comment.score)
        print('----------')
    except AttributeError:
        print(top_level_comment)
        print('----------')

In [ ]:
# Access all comments from the post (including replies to comments)
submission.comments.replace_more(limit=None)
for comment in submission.comments.list():
    print(comment.body)
    print('----------')

### Exercise 6



Pick a subreddit of your choice and:


In [ ]:
subreddit = reddit.subreddit('TrueOffMyChest')


1. Create a dataframe containing title, score, id, subreddit, url, num_comments, body, time_created for the top 100 posts of all time of that subreddit.


In [ ]:
posts = subreddit.top('all', limit=100)

def posts_to_df(posts):
    posts_data = []
    for post in posts:
        title = post.title
        score = post.score
        id = post.id
        subreddit = str(post.subreddit)
        url = post.url
        num_comments = post.num_comments
        body = post.selftext
        time_created = post.created
        
        posts_data += [(title, score, id, subreddit, url, num_comments, body, time_created)]
    
    return pd.DataFrame(posts_data, columns=['title', 'score', 'id', 'subreddit', 'url', 'num_comments', 'body', 'created'])

df = posts_to_df(posts)
df


2. Split your dataframe into words (e.g. unnest_tokens) and display the 20 most frequently occurring words after you have cleaned the dataframe of common stop words. Recommended to use the extended custom stop words list from the n-grams laboratory.


In [ ]:
from tidytext import unnest_tokens

df_words = unnest_tokens(df, "word", "body")
df_words

In [ ]:
from nltk.corpus import stopwords
from siuba import *

def remove_stopwords(df):
    basic_stop_words = stopwords.words('english')
    punctuation_signs = ['\'', "''", ';', '"', ',', '.', '!', '?', '/', '[', ']', '(', ')', '“', '”', '’', '‘', '``', "--", "{", "}", "|"]
    custom_stop_words = ["a","about","above","after","again","against","ain","all","am","an","and","any","are","aren","aren't","as","at","be","because","been","before","being","below","between","both","but","by","can","couldn","couldn't","d","did","didn","didn't","do","does","doesn","doesn't","doing","don","don't","down","during","each","few","for","from","further","had","hadn","hadn't","has","hasn","hasn't","have","haven","haven't","having","he","her","here","hers","herself","him","himself","his","how","i","if","in","into","is","isn","isn't","it","it's","its","itself","just","ll","m","ma","me","mightn","mightn't","more","most","mustn","mustn't","my","myself","needn","needn't","no","nor","not","now","o","of","off","on","once","only","or","other","our","ours","ourselves","out","over","own","re","s","same","shan","shan't","she","she's","should","should've","shouldn","shouldn't","so","some","such","t","than","that","that'll","the","their","theirs","them","themselves","then","there","these","they","this","those","through","to","too","under","until","up","ve","very","was","wasn","wasn't","we","were","weren","weren't","what","when","where","which","while","who","whom","why","will","with","won","won't","wouldn","wouldn't","y","you","you'd","you'll","you're","you've","your","yours","yourself","yourselves","could","he'd","he'll","he's","here's","how's","i'd","i'll","i'm","i've","let's","ought","she'd","she'll","that's","there's","they'd","they'll","they're","they've","we'd","we'll","we're","we've","what's","when's","where's","who's","why's","would","able","abst","accordance","according","accordingly","across","act","actually","added","adj","affected","affecting","affects","afterwards","ah","almost","alone","along","already","also","although","always","among","amongst","announce","another","anybody","anyhow","anymore","anyone","anything","anyway","anyways","anywhere","apparently","approximately","arent","arise","around","aside","ask","asking","auth","available","away","awfully","b","back","became","become","becomes","becoming","beforehand","begin","beginning","beginnings","begins","behind","believe","beside","besides","beyond","biol","brief","briefly","c","ca","came","cannot","can't","cause","causes","certain","certainly","co","com","come","comes","contain","containing","contains","couldnt","date","different","done","downwards","due","e","ed","edu","effect","eg","eight","eighty","either","else","elsewhere","end","ending","enough","especially","et","etc","even","ever","every","everybody","everyone","everything","everywhere","ex","except","f","far","ff","fifth","first","five","fix","followed","following","follows","former","formerly","forth","found","four","furthermore","g","gave","get","gets","getting","give","given","gives","giving","go","goes","gone","got","gotten","h","happens","hardly","hed","hence","hereafter","hereby","herein","heres","hereupon","hes","hi","hid","hither","home","howbeit","however","hundred","id","ie","im","immediate","immediately","importance","important","inc","indeed","index","information","instead","invention","inward","itd","it'll","j","k","keep","keeps","kept","kg","km","know","known","knows","l","largely","last","lately","later","latter","latterly","least","less","lest","let","lets","like","liked","likely","line","little","'ll","look","looking","looks","ltd","made","mainly","make","makes","many","may","maybe","mean","means","meantime","meanwhile","merely","mg","might","million","miss","ml","moreover","mostly","mr","mrs","much","mug","must","n","na","name","namely","nay","nd","near","nearly","necessarily","necessary","need","needs","neither","never","nevertheless","new","next","nine","ninety","nobody","non","none","nonetheless","noone","normally","nos","noted","nothing","nowhere","obtain","obtained","obviously","often","oh","ok","okay","old","omitted","one","ones","onto","ord","others","otherwise","outside","overall","owing","p","page","pages","part","particular","particularly","past","per","perhaps","placed","please","plus","poorly","possible","possibly","potentially","pp","predominantly","present","previously","primarily","probably","promptly","proud","provides","put","q","que","quickly","quite","qv","r","ran","rather","rd","readily","really","recent","recently","ref","refs","regarding","regardless","regards","related","relatively","research","respectively","resulted","resulting","results","right","run","said","saw","say","saying","says","sec","section","see","seeing","seem","seemed","seeming","seems","seen","self","selves","sent","seven","several","shall","shed","shes","show","showed","shown","showns","shows","significant","significantly","similar","similarly","since","six","slightly","somebody","somehow","someone","somethan","something","sometime","sometimes","somewhat","somewhere","soon","sorry","specifically","specified","specify","specifying","still","stop","strongly","sub","substantially","successfully","sufficiently","suggest","sup","sure","take","taken","taking","tell","tends","th","thank","thanks","thanx","thats","that've","thence","thereafter","thereby","thered","therefore","therein","there'll","thereof","therere","theres","thereto","thereupon","there've","theyd","theyre","think","thou","though","thoughh","thousand","throug","throughout","thru","thus","til","tip","together","took","toward","towards","tried","tries","truly","try","trying","ts","twice","two","u","un","unfortunately","unless","unlike","unlikely","unto","upon","ups","us","use","used","useful","usefully","usefulness","uses","using","usually","v","value","various","'ve","via","viz","vol","vols","vs","w","want","wants","wasnt","way","wed","welcome","went","werent","whatever","what'll","whats","whence","whenever","whereafter","whereas","whereby","wherein","wheres","whereupon","wherever","whether","whim","whither","whod","whoever","whole","who'll","whomever","whos","whose","widely","willing","wish","within","without","wont","words","world","wouldnt","www","x","yes","yet","youd","youre","z","zero","a's","ain't","allow","allows","apart","appear","appreciate","appropriate","associated","best","better","c'mon","c's","cant","changes","clearly","concerning","consequently","consider","considering","corresponding","course","currently","definitely","described","despite","entirely","exactly","example","going","greetings","hello","help","hopefully","ignored","inasmuch","indicate","indicated","indicates","inner","insofar","it'd","keep","keeps","novel","presumably","reasonably","second","secondly","sensible","serious","seriously","sure","t's","third","thorough","thoroughly","three","well","wonder"]

    stop_words = basic_stop_words + punctuation_signs + custom_stop_words
    
    df = df[~df['word'].isin(stop_words)]
    return df
    

df_clean = remove_stopwords(df_words)
df_count = count(df_clean, _['word'], sort=True)
df_count.head(20)


Pick 4 subreddits of your choice and:


In [ ]:
subreddits = ['tifu', 'depression', 'AmItheAsshole', 'legaladvice']


3. Create a dataframe containing title, score, id, subreddit, url, num_comments, body, time_created for the top 100 posts of all time of each subreddit.


In [ ]:
posts = list(map(lambda x: reddit.subreddit(x).top('all', limit=100), subreddits))

dfs = pd.concat(list(map(posts_to_df, posts)))
dfs


4. Split your dataframe into words (e.g. unnest_tokens) find the top 10 most representative words from each subreddit based on tf_idf.


In [ ]:
dfs_words = unnest_tokens(dfs, 'word', 'body')
dfs_words

In [ ]:
from tidytext import bind_tf_idf
from plotnine import *
from siuba.dply.forcats import fct_reorder

dfs_count = count(dfs_words, _.word, _.subreddit, sort=True)
dfs_tf_idf = bind_tf_idf(dfs_count, 'word', 'subreddit', 'n')

dfs_tf_idf = ungroup(group_by(arrange(dfs_tf_idf, -_.tf_idf), 'subreddit').head(10))

dfs_tf_idf = arrange(dfs_tf_idf, _.subreddit, -_.tf_idf)


ggplot(dfs_tf_idf) \
    + aes(x=fct_reorder(dfs_tf_idf['word'], x=dfs_tf_idf['tf_idf']), y='tf_idf', fill='subreddit') \
    + coord_flip() \
    + geom_col(show_legend=False) \
    + facet_wrap('~subreddit', ncol = 2, scales = "free") \
    + labs(x = "tf-idf", y = None) \
    + theme(subplots_adjust={'wspace': 0.85, 'hspace': 0.3}) \
    + scale_x_discrete()



Pick any post of your choice and:



5. Do any exploration you find suitable, anything goes. (probably worth making use of the comments from the post)




In [ ]:
# Write your code below
from tidytext import *